In [1]:
!pip install -q deepchem tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 552.4/552.4 kB 10.3 MB/s eta 0:00:00


In [3]:
import deepchem as dc
import pandas as pd
import numpy as np

# ==========================================
# 1. SET YOUR FILENAMES HERE
# ==========================================
# Make sure both of these files are uploaded to your new notebook!
TRAINING_FILE = "/content/cyp19a1_cid_smiles_cache.csv"  # Your original aromatase dataset
HERBAL_FILE = "/content/oroxylum_gnn_ready.csv"          # Your scraped plant dataset

# ==========================================
# 2. RETRAIN THE GNN (Fixes Keras mismatch)
# ==========================================
print("--- STEP 1: Loading Training Data ---")
df_train = pd.read_csv(TRAINING_FILE).dropna(subset=["SMILES"]).reset_index(drop=True)

# Define active vs inactive based on the median
potency_col = 'Activity_Value' if 'Activity_Value' in df_train.columns else df_train.select_dtypes(include=[np.number]).columns[0]
mid_thresh = df_train[potency_col].median()
df_train['target'] = (df_train[potency_col] <= mid_thresh).astype(int)

# Featurize training data
featurizer = dc.feat.ConvMolFeaturizer()
X_train = featurizer.featurize(df_train['SMILES'].tolist())
valid_train_idx = [i for i, g in enumerate(X_train) if g is not None]
X_train_clean = [X_train[i] for i in valid_train_idx]
y_train_clean = df_train['target'].iloc[valid_train_idx].values

train_dataset = dc.data.NumpyDataset(X=np.array(X_train_clean, dtype=object), y=y_train_clean)

print("Training the fresh GNN model...")
model = dc.models.GraphConvModel(n_tasks=1, mode='classification', model_dir="./new_gcn_model")
model.fit(train_dataset, nb_epoch=20) # Adjust epochs if needed
print("Training complete!")

# ==========================================
# 3. VIRTUAL SCREENING (Predicting the Herbs)
# ==========================================
print("\n--- STEP 2: Running Virtual Screening on Herbal Compounds ---")
df_herbal = pd.read_csv(HERBAL_FILE).dropna(subset=["SMILES"]).reset_index(drop=True)

# Featurize the plant molecules
X_herbal = featurizer.featurize(df_herbal['SMILES'].tolist())
valid_herbal_idx = [i for i, g in enumerate(X_herbal) if g is not None]
X_herbal_clean = [X_herbal[i] for i in valid_herbal_idx]
df_herbal_clean = df_herbal.iloc[valid_herbal_idx].reset_index(drop=True)

herbal_dataset = dc.data.NumpyDataset(X=np.array(X_herbal_clean, dtype=object))

# Predict scores (Probability of being an active inhibitor)
print("Predicting binding potential...")
probs = model.predict(herbal_dataset)[:, 0, 1]
df_herbal_clean['Predicted_Activity'] = probs

# Sort and save results
results = df_herbal_clean.sort_values(by='Predicted_Activity', ascending=False)
results.to_csv("aromatase_herbal_predictions.csv", index=False)

print("\n=== SCREENING SUCCESSFUL ===")
print("Top 5 Indian Herbal Candidates for Aromatase Inhibition:")
print(results[['Phytochemical name', 'Predicted_Activity']].head())

--- STEP 1: Loading Training Data ---


Streaming output truncated to the last 5000 lines.
[10:33:02] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[10:33:02] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[10:33:02] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[10:33:02] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[10:33:02] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[10:33:02] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[10:33:02] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[10:33:02] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[10:33:02] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[10:33:02] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[10:33:02] DEPRECATION WARNING: please use GetValence(Chem.

Training the fresh GNN model...
Training complete!

--- STEP 2: Running Virtual Screening on Herbal Compounds ---
Predicting binding potential...


[10:35:36] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[10:35:36] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[10:35:36] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[10:35:36] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[10:35:36] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[10:35:36] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[10:35:36] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[10:35:36] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[10:35:36] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[10:35:36] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[10:35:36] DEPRECATION WARNING: please use GetValence(Chem.ValenceType.IMPLICIT) instead
[10:35:36] DEPRECATIO


=== SCREENING SUCCESSFUL ===
Top 5 Indian Herbal Candidates for Aromatase Inhibition:
        Phytochemical name  Predicted_Activity
19  Dimethyl terephthalate            0.999701
12           Anthraquinone            0.998412
32           Octanoic acid            0.996744
23           Octanoic acid            0.996744
22             Lauric acid            0.978522


In [5]:
from rdkit import Chem
from rdkit.Chem import AllChem
import pandas as pd

# 1. Load your fresh predictions
df_results = pd.read_csv("/content/aromatase_herbal_predictions.csv")

# 2. Clean duplicates (like Octanoic acid appearing twice)
df_unique = df_results.drop_duplicates(subset=['Phytochemical name'])

# 3. Take the top 4 unique candidates
top_hits = df_unique.head(4)

print("Generating 3D geometries for top candidates...")

for index, row in top_hits.iterrows():
    name = row['Phytochemical name'].replace(" ", "_")
    smiles = row['SMILES']
    score = row['Predicted_Activity']

    # Convert 2D SMILES to an RDKit Molecule object
    mol = Chem.MolFromSmiles(smiles)
    # Add implicit Hydrogens (crucial for accurate docking physics)
    mol = Chem.AddHs(mol)

    # Force the molecule into a 3D conformation
    AllChem.EmbedMolecule(mol, AllChem.ETKDG())
    # Optimize the structure to its lowest energy state (MMFF94 force field)
    AllChem.MMFFOptimizeMolecule(mol)

    # Save as a standard 3D SDF file
    filename = f"{name}_top_hit.sdf"
    writer = Chem.SDWriter(filename)
    writer.write(mol)
    writer.close()

    print(f"-> Generated 3D file: {filename} (GNN Prob: {score:.4f})")

Generating 3D geometries for top candidates...
-> Generated 3D file: Dimethyl_terephthalate_top_hit.sdf (GNN Prob: 0.9997)
-> Generated 3D file: Anthraquinone_top_hit.sdf (GNN Prob: 0.9984)
-> Generated 3D file: Octanoic_acid_top_hit.sdf (GNN Prob: 0.9967)
-> Generated 3D file: Lauric_acid_top_hit.sdf (GNN Prob: 0.9785)


In [6]:
# Install OpenBabel for handling chemical file conversions
!apt-get update -qq
!apt-get install -y -qq openbabel

# Download the official, pre-compiled Linux binary for AutoDock Vina 1.2.5
!wget -q https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.5/vina_1.2.5_linux_x86_64 -O vina
!chmod +x vina
print("Environment tools successfully installed.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libboost-iostreams1.74.0:amd64.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../libboost-iostreams1.74.0_1.74.0-14ubuntu3_amd64.deb ...
Unpacking libboost-iostreams1.74.0:amd64 (1.74.0-14ubuntu3) ...
Selecting previously unselected package libinchi1.
Preparing to unpack .../libinchi1_1.03+dfsg-4_amd64.deb ...
Unpacking libinchi1 (1.03+dfsg-4) ...
Selecting previously unselected package libmaeparser1:amd64.
Preparing to unpack .../libmaeparser1_1.2.4-1build1_amd64.deb ...
Unpacking libmaeparser1:amd64 (1.2.4-1build1) ...
Selecting previously unselected package libopenbabel7.
Preparing to unpack .../libopenbabel7_3.1.1+dfsg-6ubuntu5_amd64.deb ...
Unpacking libopenbabel7 (3.1.1+dfsg-6ubuntu5) ...
Selecting previousl

In [7]:
import urllib.request

print("Downloading human Aromatase crystal structure (3EQM)...")
url = "https://files.rcsb.org/download/3EQM.pdb"
urllib.request.urlretrieve(url, "3EQM.pdb")

# Clean the PDB: Isolate Chain A and keep the essential HEME group
with open("3EQM.pdb", "r") as infile, open("receptor_clean.pdb", "w") as outfile:
    for line in infile:
        if line.startswith("ATOM") and line[21] == "A":
            outfile.write(line)
        elif line.startswith("HETATM") and "HEM" in line:
            outfile.write(line)

print("Receptor isolated. Converting to Vina-compatible PDBQT format...")
# Use OpenBabel to assign gasteiger charges and export the rigid receptor PDBQT
!obabel receptor_clean.pdb -O receptor.pdbqt -xr
print("Receptor ready for docking!")

Receptor isolated. Converting to Vina-compatible PDBQT format...
*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is receptor_clean.pdb)

1 molecule converted
Receptor ready for docking!


In [8]:
import os
import glob

# Find all 3D conformer files generated previously
sdf_files = glob.glob("*_top_hit.sdf")

print(f"Preparing {len(sdf_files)} ligands for docking...")
for sdf in sdf_files:
    pdbqt = sdf.replace(".sdf", ".pdbqt")
    # Convert and automatically calculate partial charges
    os.system(f"obabel {sdf} -O {pdbqt}")
    print(f"-> Converted {sdf} to {pdbqt}")

Preparing 4 ligands for docking...
-> Converted Dimethyl_terephthalate_top_hit.sdf to Dimethyl_terephthalate_top_hit.pdbqt
-> Converted Lauric_acid_top_hit.sdf to Lauric_acid_top_hit.pdbqt
-> Converted Anthraquinone_top_hit.sdf to Anthraquinone_top_hit.pdbqt
-> Converted Octanoic_acid_top_hit.sdf to Octanoic_acid_top_hit.pdbqt


In [12]:
import subprocess
import glob

# Confirmed literature grid coordinates targeting the 3EQM active site
center_x, center_y, center_z = 85.027, 54.737, 46.428
size_x, size_y, size_z = 22.0, 22.0, 22.0

pdbqt_ligands = glob.glob("*_top_hit.pdbqt")

print("=== STARTING AUTODOCK VINA SIMULATION ===")
for ligand in pdbqt_ligands:
    output_name = ligand.replace(".pdbqt", "_docked.pdbqt")
    log_name = ligand.replace(".pdbqt", "_vina.log")

    # FIXED: Replaced '--log {log_name}' with standard shell redirection '> {log_name}'
    cmd = (
        f"./vina --receptor receptor.pdbqt --ligand \"{ligand}\" "
        f"--center_x {center_x} --center_y {center_y} --center_z {center_z} "
        f"--size_x {size_x} --size_y {size_y} --size_z {size_z} "
        f"--out {output_name} --exhaustiveness 8 > {log_name}"
    )

    print(f"Processing docking physics for: {ligand}...")
    subprocess.run(cmd, shell=True)

print("=== DOCKING SIMULATION COMPLETE ===")

=== STARTING AUTODOCK VINA SIMULATION ===
Processing docking physics for: Dimethyl_terephthalate_top_hit.pdbqt...
Processing docking physics for: Octanoic_acid_top_hit.pdbqt...
Processing docking physics for: Anthraquinone_top_hit.pdbqt...
Processing docking physics for: Lauric_acid_top_hit.pdbqt...
=== DOCKING SIMULATION COMPLETE ===


In [13]:
import os
import glob

print("Final Ranked In Silico Docking Results:\n")
results_summary = []
pdbqt_ligands = glob.glob("*_top_hit.pdbqt")

for ligand in pdbqt_ligands:
    log_name = ligand.replace(".pdbqt", "_vina.log")
    if os.path.exists(log_name):
        with open(log_name, "r") as f:
            for line in f:
                parts = line.split()
                # Safely capture the first binding mode row
                if parts and parts[0] == "1":
                    try:
                        affinity = float(parts[1])
                        comp_name = ligand.replace("_top_hit.pdbqt", "").replace("_", " ")
                        results_summary.append((comp_name, affinity))
                        break # Only capture the top pose
                    except ValueError:
                        continue

# Sort from strongest binding (most negative) to weakest
results_summary.sort(key=lambda x: x[1])

print(f"{'Phytochemical Lead':<30} | {'Binding Affinity (kcal/mol)':<30}")
print("-" * 65)
for name, score in results_summary:
    print(f"{name:<30} | {score:<30} kcal/mol")

Final Ranked In Silico Docking Results:

Phytochemical Lead             | Binding Affinity (kcal/mol)   
-----------------------------------------------------------------
Anthraquinone                  | -9.384                         kcal/mol
Dimethyl terephthalate         | -6.807                         kcal/mol
Lauric acid                    | -6.187                         kcal/mol
Octanoic acid                  | -5.437                         kcal/mol
